In [1]:
import time
# autoreload
%load_ext autoreload
%autoreload 2

#from scipy import signal
#from scipy import interpolate
#from scipy import ndimage
import numpy as np
#import pycatch22 
#from sktime.transformations.panel import catch22
#import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
import random
load_dotenv = dotenv.load_dotenv('../.env')

# load local library
from timex import clustering
from timex import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

import datetime
from time import sleep

from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import json



In [2]:
AKI_PATH = os.environ['AKI_PATH_NEW']
os.chdir(AKI_PATH)

In [3]:
AKI_PATH


'T:\\lab_research\\RES-Folder-UPOD\\NOSTRADAMUS_SALTRO\\E_ResearchData\\2_ResearchData\\CLEANED_for_Methods_paper_longitudinal_analysis\\28102025\\ANALYSIS\\BRAM'

In [4]:
os.listdir('.')

['AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR180d_7304pts.xlsx',
 'AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR30d_7304pts.xlsx',
 'AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR60d_7304pts.xlsx',
 'AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR90d_7304pts.xlsx',
 'AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SWnond_TR0d_7304pts.xlsx',
 'analysed_DV_LCMM_outcome_TR0d.parquet',
 'analysed_DV_LCMM_outcome_TR180d.parquet',
 'analysed_DV_LCMM_outcome_TR30d.parquet',
 'analysed_DV_LCMM_outcome_TR60d.parquet',
 'analysed_DV_LCMM_outcome_TR90d.parquet',
 'cleaned_DV_LCMM_data.parquet',
 'Results',
 '~$AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR180d_7304pts.xlsx',
 '~$AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR30d_7304pts.xlsx',
 '~$AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR60d_7304pts.xlsx',
 '~$AIC_BIC_discr_entropy_runtime_table_FU0-3650d_SW365d_TR90d_7304pts.xlsx',
 '~$AIC_BIC_discr_entropy_runtime_table_FU0-365

In [5]:
ts_data = pd.read_parquet(os.path.join(AKI_PATH, 'cleaned_DV_LCMM_data.parquet'))
ts_data = ts_data.rename(columns={"Time_since_index_FU_days": "Time_days"})
ts_data.ID = ts_data.ID.astype('int64')
ts_data['dataset_nr'] = ts_data['dataset_nr'].fillna(-1).astype('int64')


In [6]:
ts_data.groupby('dataset_nr').ID.nunique(), ts_data.ID.nunique()

(dataset_nr
 -1    14359
  1     1000
  2     1000
  3     1000
  4     1000
  5     1000
  6     1000
  7     1000
  8      304
 Name: ID, dtype: int64,
 21663)

In [ ]:
MIN_TIME = 365 * 1 # days
MAX_TIME = 365 * 10 # days
MIN_MEAS_COUNT = 3 # measurements
INTERP_RES = 1
SMOOTHING_WINDOW = 365 # in days: 4 * INTERP_RES = 360
SMOOTHING_TYPE = 'gaussian_kernel'  # 'gaussian_kernel' or 'rolling_mean'
META_KEYS = ['ID', 'Time_days']
RAW_VAL_COL = 'eGFRcr_CKDEpi2009'
INT_VAL_COL = 'eGFR_int'
SM30_VAL_COL = 'eGFR_SW30'
SM365_VAL_COL = 'eGFR_SW365'

DS_SELECTION =  [list(range(1,K+1)) for K in range(1,9)]  # which datasets to include in the analysis
SELECTION_RES = [180]
CLUSTER_NUMS =  [2, 4 , 6, 8, 10, 12, 14, 16]

CLUSTERING_ALGO='gmm'
CLUSTER_KWARGS={"reg_covar": 1e-5, "covariance_type": "diag"}

ADD_TS_META = True
EXTRACTORS = ['custom', 'catch22']



In [8]:
ts_data_df = ts_data[['ID', 'Time_days', 'eGFRcr_CKDEpi2009', 'dataset_nr']].dropna(subset=['eGFRcr_CKDEpi2009'])

In [9]:
file_dir = f"Results/{"_".join(EXTRACTORS)}"
if not os.path.exists(file_dir):
    os.makedirs(file_dir)

In [10]:
for NUM_CLUSTERS in CLUSTER_NUMS:
    for SEL_RES in SELECTION_RES:
        for DS_SEL in DS_SELECTION:
            print(f'Running clustering for DS{DS_SEL}_C{NUM_CLUSTERS}_TR{SEL_RES}')


            Sel_IDS = ts_data[ts_data['dataset_nr'].isin(DS_SEL)].ID.unique()
            ts_data_run = ts_data_df[ts_data_df.ID.isin(Sel_IDS)]

            SETTINGS_DICT = {
                'MIN_TIME': MIN_TIME,
                'MAX_TIME': MAX_TIME,
                'MIN_MEAS_COUNT': MIN_MEAS_COUNT,
                'INTERP_RES': INTERP_RES,
                'SMOOTHING_WINDOW': SMOOTHING_WINDOW,
                'SMOOTHING_TYPE': SMOOTHING_TYPE,
                'META_KEYS': META_KEYS,
                'RAW_VAL_COL': RAW_VAL_COL,
                'INT_VAL_COL': INT_VAL_COL,
                'SM30_VAL_COL': SM30_VAL_COL,
                'SM365_VAL_COL': SM365_VAL_COL,
                'DS_SELECTION': DS_SEL,
                'SELECTION_RES': SEL_RES,
                'NUM_CLUSTERS': NUM_CLUSTERS,
                'CLUSTERING_ALGO': CLUSTERING_ALGO,
                'CLUSTER_KWARGS': CLUSTER_KWARGS,
                'ADD_TS_META': ADD_TS_META,
                'EXTRACTORS': EXTRACTORS
            }

            ts_clusterer = clustering.CrossSectionalClustering(smoothing=True, 
                                                            smoothing_type=SMOOTHING_TYPE,
                                                            smoothing_window_size=SMOOTHING_WINDOW,
                                                            n_skip=3,
                                                            interpolation=True, 
                                                            interpolation_resolution=INTERP_RES,
                                                            interpolation_keep_init=True,
                                                            analysis_resolution=SEL_RES,
                                                            min_measurements_per_id=MIN_MEAS_COUNT, 
                                                            min_time=MIN_TIME,
                                                            max_time=MAX_TIME,
                                                            clustering_algorithm=CLUSTERING_ALGO,
                                                            n_clusters=NUM_CLUSTERS, 
                                                            cluster_kwargs=CLUSTER_KWARGS,
                                                            id_column='ID', 
                                                            time_column='Time_days',
                                                            feature_columns=[RAW_VAL_COL],
                                                            imputation_method='knn',
                                                            cross_standardisation=True,
                                                            normalise_timeseries= "group",
                                                            normalisation_method="standard",
                                                            add_ts_meta=ADD_TS_META,
                                                            extractors=EXTRACTORS,
                                                            verbose=True)


            ts_clusterer.fit(ts_data_run)


            ts_label_df = pd.read_parquet(f'analysed_DV_LCMM_outcome_TR{SEL_RES}d.parquet')
            ts_label_df['ID'] = ts_label_df.ID.astype(int)
            ts_label_df.set_index('ID', inplace=True)
            ts_label_df.dropna(how='all', inplace=True)

            try:
                class_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SEL_RES}_Class' 
                proba_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SEL_RES}_max_prob'    
                ts_label_df_LCMM = ts_label_df.dropna(subset=[class_string])[[proba_string, class_string]]
                ts_label_df_LCMM[class_string] = ts_label_df_LCMM[class_string].astype('int')
                
                ts_label_df_LCMM = ts_label_df_LCMM.rename(columns={
                    proba_string: 'LCMM_max_prob',
                    class_string: 'LCMM_Class'
                    })

                res_cluster = pd.DataFrame(zip(ts_clusterer.ts_cross_combined.index, ts_clusterer.predict()), columns=['ID', 'cluster'])
                res_cluster['cluster'] = res_cluster['cluster'].astype(int)

                res_cluster_proba = pd.DataFrame()
                res_cluster_proba['ID'] = ts_clusterer.ts_cross_combined.index
                res_cluster_proba[[f'cluster_prob_{i}' for i in range(NUM_CLUSTERS)]] =  ts_clusterer.predict_proba()

                res_final = ts_data_run.merge(res_cluster, how='left', left_on='ID', right_on='ID').dropna(subset='ID')
                res_final = res_final.merge(ts_label_df_LCMM, how='left', left_on='ID', right_index=True).dropna(subset=['cluster'])\
                                        .astype({'cluster': 'int'})

                res_final_ = res_final.groupby('ID')[['LCMM_Class', 'cluster']].first()
                res_final_['LCMM_Class'] = res_final_['LCMM_Class'].astype(int) - 1

                external_scores = {
                    'ari': adjusted_rand_score(res_final_['cluster'], res_final_['LCMM_Class']), 
                    'ami': adjusted_mutual_info_score(res_final_['cluster'], res_final_['LCMM_Class'])
                }
            except Exception as e:
                print(f'Could not compute external scores: {e}')
                external_scores = {
                    'ari': None,
                    'ami': None
                }


            internal_scores = ts_clusterer.get_scores()

            # combine all scores and timings in a single dictionary
            scores_dict = {
                'external_scores': external_scores,
                'internal_scores': internal_scores,
                'timings': ts_clusterer.timings,
                'settings': SETTINGS_DICT
            }

            # write scores to a jsonl file, appending if the file already exists
            with open(f'Results/{"_".join(EXTRACTORS)}clustering_scores_log.jsonl', 'a') as f:
                f.write(json.dumps(scores_dict) + '\n') 


            res_cluster_proba.to_csv(f'Results/{"_".join(EXTRACTORS)}/cluster_probs_ds{''.join([str(c) for c in DS_SEL])}_C{NUM_CLUSTERS}_TR{SEL_RES}.csv', sep=';')

2025-12-01 14:26:52,601 - timex.clustering - DEBUG - CrossSectionalClustering initialized
2025-12-01 14:26:52,601 - timex.clustering - INFO - Starting CrossSectionalClustering.fit(); TS shape (30127, 4)


Running clustering for DS[1]_C2_TR180
eGFRcr_CKDEpi2009


2025-12-01 14:26:52,601 - timex.clustering - INFO - Processing feature column: eGFRcr_CKDEpi2009
2025-12-01 14:26:52,601 - timex.clustering - INFO - Filtering completed for eGFRcr_CKDEpi2009 in 0.0035 seconds. TS: (30031, 3)
100%|██████████| 968/968 [00:00<00:00, 1032.92it/s]
2025-12-01 14:26:55,979 - timex.clustering - INFO - Interpolation completed for eGFRcr_CKDEpi2009 in 3.3665 seconds, TS: (3533200, 3)
100%|██████████| 968/968 [00:05<00:00, 184.06it/s]
2025-12-01 14:27:03,615 - timex.clustering - INFO - Smoothing completed for eGFRcr_CKDEpi2009 in 7.6349 seconds, TS: (3533200, 3)
2025-12-01 14:27:03,975 - timex.clustering - INFO - Selection after smoothing for eGFRcr_CKDEpi2009 in 7.6349 seconds, TS: (20328, 3)
2025-12-01 14:27:03,978 - timex.clustering - INFO - Selection after pruning NaNs for eGFRcr_CKDEpi2009 in 0.0016 seconds, TS: (10732, 3)
100%|██████████| 968/968 [00:01<00:00, 845.72it/s]
2025-12-01 14:27:05,132 - timex.clustering - INFO - Normalization completed for eGFRcr

Processing custom features..


  0%|          | 0/968 [00:00<?, ?it/s]c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 2 is too high: all coefficients will experience boundary effects.
  warnings.warn(
\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\extractor.py:1107: RuntimeWarning: divide by zero encountered in log
  poly = np.polyfit(np.log(lags), np.log(tau), 1)
\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\extractor.py:1027: RuntimeWarning: divide by zero encountered in log2
  entropy = np.nansum(psd_norm * np.log2(psd_norm))
\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\extractor.py:1027: RuntimeWarning: invalid value encountered in multiply
  entropy = np.nansum(psd_norm * np.log2(psd_norm))
c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\numpy\_core\_methods.py:218: Runtime

...

2025-12-01 16:56:54,897 - timex.clustering - INFO - Processing feature column: eGFRcr_CKDEpi2009
2025-12-01 16:56:54,921 - timex.clustering - INFO - Filtering completed for eGFRcr_CKDEpi2009 in 0.0211 seconds. TS: (218731, 3)
100%|██████████| 7074/7074 [00:07<00:00, 899.87it/s] 
2025-12-01 16:57:20,328 - timex.clustering - INFO - Interpolation completed for eGFRcr_CKDEpi2009 in 25.4062 seconds, TS: (25820100, 3)
100%|██████████| 7074/7074 [03:20<00:00, 35.31it/s]
2025-12-01 17:00:57,916 - timex.clustering - INFO - Smoothing completed for eGFRcr_CKDEpi2009 in 217.5888 seconds, TS: (25820100, 3)
2025-12-01 17:00:59,147 - timex.clustering - INFO - Selection after smoothing for eGFRcr_CKDEpi2009 in 217.5888 seconds, TS: (148554, 3)
2025-12-01 17:00:59,151 - timex.clustering - INFO - Selection after pruning NaNs for eGFRcr_CKDEpi2009 in 0.0031 seconds, TS: (79937, 3)
100%|██████████| 7074/7074 [00:08<00:00, 811.20it/s]
2025-12-01 17:01:07,931 - timex.clustering - INFO - Normalization comple

Processing custom features..


  0%|          | 0/7074 [00:00<?, ?it/s]c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 2 is too high: all coefficients will experience boundary effects.
  warnings.warn(
c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\numpy\_core\_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\numpy\_core\_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\numpy\_core\_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
\\DS.UMCUTRECHT.NL\DATA\LAB\laupod

Processing catch22 features..


100%|██████████| 7074/7074 [00:01<00:00, 4404.49it/s]
2025-12-01 17:01:38,410 - timex.clustering - INFO - Cross-sectional extraction completed for eGFRcr_CKDEpi2009 in 30.4798 seconds, TS cross: (7074, 192)
2025-12-01 17:01:38,411 - timex.clustering - INFO - Joining completed for eGFRcr_CKDEpi2009 in 0.0000 seconds
2025-12-01 17:01:38,431 - timex.clustering - INFO - Replaced inf's by NaN's in 0.0179 seconds
2025-12-01 17:01:38,437 - timex.clustering - INFO - Removed 68 columns with more than 75.0% missingness in 0.0057 seconds
2025-12-01 17:01:38,448 - timex.clustering - INFO - Removed 5 columns with zero variance 0.0109 seconds
119it [00:00, 1645.77it/s]
2025-12-01 17:01:38,525 - timex.clustering - INFO - Removed 0 columns because of duplication in 0.0764 seconds
2025-12-01 17:01:38,548 - timex.clustering - INFO - Standardization completed in 0.0215 seconds
2025-12-01 17:01:38,551 - timex.clustering - INFO - Found 11824 missing values, starting imputation
2025-12-01 17:01:38,551 - tim

In [ ]:
# # plot 10 random samples
# for s in ts_clusterer.ts_filtered.sample(n=1)['ID']:
#     tsv = ts_clusterer.ts_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='red', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='green', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='blue', alpha=0.7)

: 